# Multi-factor designs — ASCA and linear mixed models

Most metabolomics studies are **not** two-group comparisons. A typical
design might be `treatment × time` with patient IDs (repeated
measures). The two-group `metabol.differential` test is the wrong tool
here.

- **`ov.metabol.asca`** (Smilde 2005) — ANOVA-Simultaneous Component
  Analysis. Decomposes the data into per-factor effect matrices plus
  pairwise interactions, runs PCA on each, and reports
  variance-explained + permutation p-value.
- **`ov.metabol.mixed_model`** — per-feature `statsmodels.MixedLM`
  with a user-defined formula and a random-effect grouping variable
  (e.g. patient ID).

This tutorial uses a **synthetic 2×2 factorial** (treatment × time,
6 patients × 4 cells = 24 samples × 25 features) with:

- Strong treatment effect on features 0–4
- Moderate time effect on features 5–9
- Per-patient random intercept


## 0 — Setup and synthetic factorial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from anndata import AnnData
import omicverse as ov

rng = np.random.default_rng(1)

n_per_cell = 6  # patients
treatments = ['ctrl', 'drug']
times = ['0h', '24h']
n_features = 25

rows, idx = [], []
for t in treatments:
    for tm in times:
        for k in range(n_per_cell):
            rows.append({'treatment': t, 'time': tm, 'patient': f'p{k}'})
            idx.append(f'{t}_{tm}_{k}')
obs = pd.DataFrame(rows, index=idx)
n = len(obs)

X = rng.standard_normal((n, n_features)) * 0.3
tmask = (obs['treatment'] == 'drug').to_numpy()
tm_mask = (obs['time'] == '24h').to_numpy()
X[tmask, 0:5] += 2.0       # treatment effect
X[tm_mask, 5:10] += 1.0    # time effect

# Per-patient random intercept (what MixedLM will estimate)
intercepts = rng.standard_normal(n_per_cell) * 1.0
for i, p in enumerate(obs['patient']):
    X[i, :] += intercepts[int(p[1:])]

var = pd.DataFrame(index=[f'feat{i}' for i in range(n_features)])
adata = AnnData(X=X, obs=obs, var=var)
adata.shape


## 1 — ASCA

Decompose into `treatment`, `time`, `treatment:time` interaction and
residual. Use 500 permutations for significance testing.


In [ ]:
res = ov.metabol.asca(
    adata,
    factors=['treatment', 'time'],
    include_interactions=True,
    n_components=2,
    n_permutations=500,
    seed=0,
)
res.summary()


### Scores plot for the treatment effect

The scores matrix of the `treatment` effect places each sample on a
2-D projection of the factor's subspace. Samples of the same level
should cluster tightly.


In [ ]:
scores = res.scores_frame('treatment')
fig, ax = plt.subplots(figsize=(6, 4))
for lvl, colour in zip(['ctrl', 'drug'], ['C0', 'C3']):
    mask = (adata.obs['treatment'] == lvl).to_numpy()
    ax.scatter(scores.loc[mask, 'PC1'], scores.loc[mask, 'PC2'],
               label=lvl, s=50, edgecolor='k')
ax.set_xlabel('PC1 (treatment effect)')
ax.set_ylabel('PC2 (treatment effect)')
ax.legend(title='treatment')
fig.tight_layout()
plt.show()


### Loadings — which features drive the treatment effect?

In [ ]:
load = res.loadings_frame('treatment')
top_treat = load['PC1'].abs().sort_values(ascending=False).head(8)
print('Top treatment-driving features (|PC1 loading|):')
print(top_treat)


## 2 — Linear mixed model

For a specific per-feature effect size + p-value, fit `MixedLM` with
`treatment + time` as fixed effects and `patient` as the random
intercept. Ask for the `treatment[T.drug]` contrast in short format
to get the same schema as `metabol.differential`.


In [ ]:
tbl = ov.metabol.mixed_model(
    adata,
    formula='treatment + time',
    groups='patient',
    term='treatment[T.drug]',
)
tbl.sort_values('pvalue').head(10)


The top features should match the planted-signal block (0–4).
FDR correction is applied automatically (`padj` column).


In [ ]:
n_sig = (tbl['padj'] < 0.05).sum()
print(f'{n_sig} features with padj < 0.05 for treatment[T.drug]')


## Takeaways

- `asca` — global structure; answers *which factor explains overall
  variance?*
- `mixed_model` — per-feature effect size + p; answers *which
  metabolites change with treatment while respecting patient
  identity?*

Use them together: ASCA to choose which factor is worth testing,
MixedLM to get publication-ready per-feature statistics.
